# RAG Policy Comparator
Upload **two policy documents** (PDF / DOCX / TXT) and ask questions about how they differ. Answers are generated **only from retrieved text**, with citations.

**How it works:** parse → section-aware chunking → embeddings → retrieve top-k chunks **from each document separately** (so one doc can't crowd out the other) → a small open-source LLM (runs inside Colab, no API key) answers **using only that context**. Weak matches are dropped, 'not found' is returned when nothing relevant is retrieved, and every answer is checked for numbers or citations that don't appear in the retrieved text.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free). CPU works but is slow.

**Run order:** 1 install → 2 load local model → 3 upload PDFs → 4 RAG engine → 5 ask → 6 optional test suite.

### 1. Install

In [ ]:
!pip install -q pypdf python-docx sentence-transformers transformers accelerate scikit-learn

### 2. Load a small free local LLM
No API key needed. Downloads **Qwen2.5-1.5B-Instruct** (~3 GB, one-time, about a minute) and runs inside your Colab session. Works on the free T4 GPU and, more slowly, on CPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GPU = torch.cuda.is_available()
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct" if GPU else "Qwen/Qwen2.5-0.5B-Instruct"   # 0.5B only as a CPU fallback
if not GPU:
    print("WARNING: no GPU detected -> slow. Runtime > Change runtime type > T4 GPU, then Runtime > Restart and run all.")
print("GPU:", GPU, "| model:", MODEL_ID)
if not GPU: print("WARNING: no GPU - answers will be 10-20x slower. Runtime > Change runtime type > T4 GPU, then re-run from cell 1.")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if GPU else torch.float32,
    device_map="auto" if GPU else None,
)
model.eval()

def generate(system, user, max_new_tokens=200):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

import time; t = time.time(); print(generate("You are helpful.", "Say OK.")); print(f"warm-up: {time.time()-t:.1f}s on", "GPU" if GPU else "CPU")

### 3. Upload your two policy documents
Select **both files** in the upload dialog (PDF, DOCX or TXT). The first (alphabetically) becomes Document A, the second Document B.

In [ ]:
from google.colab import files
up = files.upload()
names = sorted(up.keys())
assert len(names) == 2, f"Please upload exactly 2 files (got {len(names)})"
assert all(n.lower().endswith((".pdf", ".docx", ".txt")) for n in names), "Use PDF, DOCX or TXT files"
PATH_A, PATH_B = names
print("Document A:", PATH_A, "\nDocument B:", PATH_B)

### 4. RAG engine

In [ ]:
import re, os, numpy as np
from pypdf import PdfReader

# ---------- 1. Load ----------
def load_text(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return "\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    if ext == ".docx":
        import docx
        return "\n".join(p.text for p in docx.Document(path).paragraphs)
    return open(path, encoding="utf-8", errors="ignore").read()

# ---------- 2. Chunk (section-aware) ----------
HEADING = re.compile(r"^\s*(\d+(\.\d+)*\.?|[A-Z][A-Za-z]*\.|Article\s+\d+|Section\s+\d+)\s+\S.{0,80}$")

def chunk_text(text, max_chars=900, overlap=150):
    """Split on headings; long sections are split by sentences with overlap. Each chunk keeps its heading."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    sections, head, buf = [], "Preamble", []
    for l in lines:
        if HEADING.match(l) and len(l) < 90 and not l.endswith("."):
            if buf: sections.append((head, " ".join(buf)))
            head, buf = l, []
        else:
            buf.append(l)
    if buf: sections.append((head, " ".join(buf)))

    chunks = []
    for head, body in sections:
        if len(body) <= max_chars:
            chunks.append({"section": head, "text": f"{head}: {body}"})
            continue
        sents, cur = re.split(r"(?<=[.!?])\s+", body), ""
        for s in sents:
            if len(cur) + len(s) > max_chars and cur:
                chunks.append({"section": head, "text": f"{head}: {cur.strip()}"})
                cur = cur[-overlap:] + " " + s
            else:
                cur += " " + s
        if cur.strip(): chunks.append({"section": head, "text": f"{head}: {cur.strip()}"})
    return chunks

# ---------- 3. Embed ----------
class Embedder:
    def __init__(self):
        try:
            from sentence_transformers import SentenceTransformer
            self.m = SentenceTransformer("all-MiniLM-L6-v2"); self.kind = "sentence-transformers"
        except Exception as e:
            print("sentence-transformers unavailable, using TF-IDF fallback:", e)
            self.m, self.kind = None, "tfidf"
    def fit(self, corpus):
        if self.kind == "tfidf":
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.v = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english").fit(corpus)
    def encode(self, texts):
        if self.kind == "tfidf":
            from sklearn.preprocessing import normalize
            return normalize(self.v.transform(texts)).toarray()
        return self.m.encode(texts, normalize_embeddings=True)

# ---------- 4. Index ----------
class PolicyIndex:
    def __init__(self, path_a, path_b):
        self.names = {"A": os.path.basename(path_a), "B": os.path.basename(path_b)}
        self.chunks = {"A": chunk_text(load_text(path_a)), "B": chunk_text(load_text(path_b))}
        self.emb = Embedder()
        self.emb.fit([c["text"] for k in "AB" for c in self.chunks[k]])
        self.vecs = {k: self.emb.encode([c["text"] for c in self.chunks[k]]) for k in "AB"}
        self.min_score = 0.03 if self.emb.kind == "tfidf" else 0.20   # below this a passage counts as 'not relevant'
        print(f"Indexed A: {len(self.chunks['A'])} chunks | B: {len(self.chunks['B'])} chunks | embedder: {self.emb.kind}")

    def retrieve(self, query, k=4):
        """Top-k PER DOCUMENT - guarantees both sides are represented."""
        q = self.emb.encode([query])[0]
        out = {}
        for d in "AB":
            sims = self.vecs[d] @ q
            top = np.argsort(-sims)[:k]
            out[d] = [(self.chunks[d][i], float(sims[i])) for i in top]
        return out

# ---------- 5. Generate (grounded: answers only from retrieved passages) ----------
SYSTEM = """You are a strict policy-comparison assistant. You may ONLY use facts written in the CONTEXT passages.
Rules:
- Never use outside knowledge. Never guess. Never calculate new numbers; copy numbers exactly as written.
- If the CONTEXT does not contain the answer, reply exactly: Not found in the provided documents.
- If a topic appears in only one document, write: Not mentioned in this document. for the other.
- Put the section after each fact, like [A: 3. Annual Leave].
Answer in exactly this format:
**Document A:** what A says [A: section]
**Document B:** what B says [B: section]
**Difference:** one short sentence, or No difference.
Be brief: under 70 words in total."""

NOT_FOUND = "Not found in the provided documents."

def build_context(idx, hits):
    parts = []
    for d in "AB":
        parts.append(f"=== Document {d}: {idx.names[d]} ===")
        if not hits[d]:
            parts.append("(no relevant passage found in this document)")
        for c, s in hits[d]:
            parts.append(f"[{d}: {c['section']}]\n{c['text']}")
    return "\n\n".join(parts)

def _nums(t):
    return set(n.rstrip(".,") for n in re.findall(r"\d+(?:[.,]\d+)*", t))

def answer(idx, question, k=2):
    """Retrieve -> filter weak matches -> generate from context only -> verify numbers/citations."""
    raw = idx.retrieve(question, k)
    hits = {}
    for d in "AB":   # keep only strong matches: above the floor AND close to this doc's best match -> smaller prompt, faster answer
        top = raw[d][0][1] if raw[d] else 0
        hits[d] = [(c, s) for c, s in raw[d] if s >= idx.min_score and s >= 0.6 * top]
    if not hits["A"] and not hits["B"]:          # nothing relevant retrieved -> don't call the LLM at all
        return {"answer": NOT_FOUND, "sources": [], "warnings": [], "hits": hits}
    context = build_context(idx, hits)
    ans = generate(SYSTEM, f"CONTEXT:\n{context}\n\nQUESTION: {question}")

    warnings = []
    extra = _nums(ans) - _nums(context)           # numbers the model invented or computed
    if extra:
        warnings.append(f"Numbers not found in the retrieved text: {sorted(extra)}. Verify against the sources.")
    valid = {(d, c["section"]) for d in "AB" for c, _ in hits[d]}
    bad = [f"{d}: {s}" for d, s in set(re.findall(r"\[([AB]): ([^\]]+)\]", ans)) if (d, s.strip()) not in valid]
    if bad:
        warnings.append(f"Cites sections that were not retrieved: {bad}.")
    sources = [f"{d}: {c['section']} ({s:.2f})" for d in "AB" for c, s in hits[d]]
    return {"answer": ans, "sources": sources, "warnings": warnings, "hits": hits}

def ask(idx, question, k=2, show_chunks=False):
    import time; t0 = time.time()
    r = answer(idx, question, k)
    if show_chunks:
        for d in "AB":
            print(f"\n--- Retrieved from {d} ({idx.names[d]}) ---")
            for c, s in r["hits"][d]:
                print(f"  [{s:.2f}] {c['text'][:160]}...")
    out = r["answer"]
    if r["sources"]:
        out += "\n\n---\n*Sources used:* " + " · ".join(r["sources"])
    for w in r["warnings"]:
        out += f"\n\n⚠️ {w}"
    return out + f"\n\n*⏱ {time.time() - t0:.1f}s*"

# ---------- 6. No-LLM side-by-side diff (instant, deterministic) ----------
import difflib, html
from IPython.display import HTML, display

def diff_view(idx, question, k=1):
    """Word-level highlighted diff of the best-matching passage from each doc. Red = only in A, green = only in B."""
    hits = idx.retrieve(question, k)
    for (ca, _), (cb, _) in zip(hits["A"], hits["B"]):
        a, b = ca["text"].split(), cb["text"].split()
        ha, hb = [], []
        for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b).get_opcodes():
            if tag == "equal":
                ha.append(html.escape(" ".join(a[i1:i2]))); hb.append(html.escape(" ".join(b[j1:j2])))
            else:
                if i2 > i1: ha.append(f"<span style='background:#ffc9c9;color:#000'>{html.escape(' '.join(a[i1:i2]))}</span>")
                if j2 > j1: hb.append(f"<span style='background:#c9f5c9;color:#000'>{html.escape(' '.join(b[j1:j2]))}</span>")
        display(HTML(f"<table style='width:100%;table-layout:fixed;border-collapse:collapse'><tr>"
                     f"<td style='vertical-align:top;padding:8px;border:1px solid #888'><b>A: {idx.names['A']}</b><br>{' '.join(ha)}</td>"
                     f"<td style='vertical-align:top;padding:8px;border:1px solid #888'><b>B: {idx.names['B']}</b><br>{' '.join(hb)}</td></tr></table>"))

idx = PolicyIndex(PATH_A, PATH_B)

### 5. Ask questions
Try: *What are the differences in annual leave?* · *Which document gives more sick leave?* · *What is only in one of the two documents?*

In [ ]:
from IPython.display import Markdown, display
q = "What changed in annual leave and carry-forward rules between the two documents?"
display(Markdown(ask(idx, q, show_chunks=True)))

**Instant no-LLM view** - highlights exactly which words differ in the best-matching passage:

In [ ]:
diff_view(idx, "annual leave carry forward")

In [ ]:
# Interactive loop - type a question, press Enter on an empty line to stop
while True:
    q = input("\nYour question (blank to stop): ").strip()
    if not q: break
    display(Markdown(ask(idx, q)))

### 6. Demo check (run before presenting)
Checks **retrieval** (right section returned for each doc?) and **answers** (do they contain the expected facts?). Small local models can miss a detail - "CHECK" means read that answer yourself. Built for the demo pair: **01_Acme_Leave_Remote_Policy_2023.pdf** and **02_Acme_Leave_Remote_Policy_2025.pdf**. Run it before presenting to confirm everything works.

In [ ]:
TESTS = [
 # (question, section keyword that MUST be retrieved, facts expected in the answer)
 ("How many annual leave days does each document give?",          "Annual Leave",  ["18", "24"]),
 ("What is the carry-forward limit for unused leave?",            "Annual Leave",  ["5", "10"]),
 ("When is a medical certificate needed for sick leave?",         "Sick Leave",    ["2", "3"]),
 ("How long is maternity leave in each?",                         "Maternity",     ["12", "26"]),
 ("How many remote work days per week are allowed?",              "Remote Work",   ["2", "3"]),
 ("Can employees encash unused leave when they leave?",           "Encashment",    ["10", "no longer"]),
 ("Is there a mental wellness day benefit?",                      "Mental",        ["2", "not mentioned"]),
 ("What is the notice period?",                                   "Notice Period", ["30", "60"]),
 ("Are working hours different?",                                 "Working Hours", ["9"]),
]
ok_ret = ok_ans = 0
for q, sec, facts in TESTS:
    hits = idx.retrieve(q, k=3)
    # Mental wellness exists only in B, so require it from B; others from both
    need = ["B"] if sec == "Mental" else ["A", "B"]
    r = all(any(sec.lower() in c["section"].lower() for c, _ in hits[d]) for d in need)
    ok_ret += r
    line = f"{'PASS' if r else 'FAIL'} retrieval | {q}"
    a = answer(idx, q)["answer"].lower()
    f = all(x.lower() in a for x in facts)
    ok_ans += f
    line += f"  || {'PASS' if f else 'CHECK'} answer"
    print(line)
print(f"\nRetrieval: {ok_ret}/{len(TESTS)} | Answers: {ok_ans}/{len(TESTS)}")